# Real-Time Web Search with GPT Using free-web-search-ultimate

This cookbook demonstrates how to give GPT models real-time web search capabilities using the [free-web-search-ultimate](https://github.com/wd041216-bit/free-web-search-ultimate) Python package — **completely free, no additional API keys required**.

[![free-web-search-ultimate](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate/badges/score.svg)](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)

## What you'll learn

- How to use `free-web-search-ultimate` as a function/tool for GPT models
- How to build a research assistant that can search the web in real-time
- How to handle multi-turn tool use for complex research queries
- How to combine web search results with GPT's reasoning capabilities

## Why free-web-search-ultimate?

| Feature | Details |
|---------|--------|
| **Cost** | Completely free — no API key, no subscription |
| **Privacy** | Uses DuckDuckGo and privacy-respecting search engines |
| **Reliability** | Multiple search backends with automatic fallback |
| **MCP Support** | Works as an MCP server for Claude Desktop and other clients |
| **CLI Support** | Also works as a standalone CLI tool (`free-web-search`) |

## Setup

```bash
pip install free-web-search-ultimate openai
```

In [ ]:
# Install required packages
%pip install free-web-search-ultimate openai --quiet

In [ ]:
import json
import os
from openai import OpenAI
from free_web_search.search_web import UltimateSearcher
import json

# Initialize the searcher
_searcher = UltimateSearcher()

def search(query: str, max_results: int = 5) -> list:
    """Search the web for real-time information."""
    answer = _searcher.search(query)
    return [
        {'title': s.title, 'url': s.url, 'snippet': s.snippet}
        for s in answer.sources[:max_results]
    ]

def search_news(query: str, max_results: int = 5) -> list:
    """Search for recent news articles."""
    answer = _searcher.search(query, search_type='news')
    return [
        {'title': s.title, 'url': s.url, 'snippet': s.snippet, 'date': getattr(s, 'date', '')}
        for s in answer.sources[:max_results]
    ]

# Initialize the OpenAI client
# Make sure OPENAI_API_KEY is set in your environment
client = OpenAI()

print("Setup complete!")

## Define the Web Search Tools

We'll define two tools for GPT:
1. `web_search` — General web search using DuckDuckGo
2. `news_search` — Search for recent news articles

In [ ]:
# Define the tools for GPT
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information. Use this tool when you need up-to-date information that may not be in your training data, such as recent events, current prices, latest news, or any time-sensitive information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up on the web"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to return (default: 5)",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "news_search",
            "description": "Search for recent news articles. Use this when you need the latest news on a topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The news topic to search for"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of news articles to return (default: 5)",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    }
]

def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Execute a tool and return the result as a string."""
    try:
        if tool_name == "web_search":
            results = search(
                query=tool_input["query"],
                max_results=tool_input.get("max_results", 5)
            )
        elif tool_name == "news_search":
            results = search_news(
                query=tool_input["query"],
                max_results=tool_input.get("max_results", 5)
            )
        else:
            return f"Unknown tool: {tool_name}"
        
        # Format results as a readable string
        if not results:
            return "No results found."
        
        formatted = []
        for i, result in enumerate(results, 1):
            title = result.get('title', 'No title')
            url = result.get('url', result.get('href', 'No URL'))
            body = result.get('body', result.get('snippet', 'No description'))
            formatted.append(f"{i}. **{title}**\n   URL: {url}\n   {body}")
        
        return "\n\n".join(formatted)
    except Exception as e:
        return f"Error executing {tool_name}: {str(e)}"

print("Tools defined successfully!")

## Build the Research Assistant

Now let's create a function that handles the full tool-use loop with GPT.

In [ ]:
def research_with_gpt(question: str, model: str = "gpt-4o-mini") -> str:
    """
    Use GPT with web search tools to answer a question.
    
    Args:
        question: The question to research
        model: The GPT model to use (default: gpt-4o-mini for cost efficiency)
    
    Returns:
        The final answer from GPT
    """
    messages = [
        {
            "role": "system",
            "content": "You are a helpful research assistant with access to real-time web search. "
                       "Use the web_search and news_search tools to find current information when needed. "
                       "Always cite your sources."
        },
        {
            "role": "user",
            "content": question
        }
    ]
    
    print(f"Question: {question}\n")
    
    # Agentic loop: keep calling GPT until it stops using tools
    # max_iterations prevents infinite loops if the model keeps issuing tool calls
    max_iterations = 10
    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        messages.append(message)
        
        # Check if GPT wants to use tools
        if message.tool_calls:
            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_input = json.loads(tool_call.function.arguments)
                
                print(f"Using tool: {tool_name}")
                print(f"Query: {tool_input.get('query', '')}")
                
                # Execute the tool
                result = execute_tool(tool_name, tool_input)
                
                # Add tool result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        else:
            # No more tool calls — return the final answer
            print("\n--- Final Answer ---")
            return message.content
    # Fallback if max iterations reached
    return "Max iterations reached. Last response: " + (message.content or "No response")

print("Research assistant ready!")

## Example 1: Current Events Research

Let's ask GPT about recent news that it couldn't know from training data alone.

In [ ]:
# Example 1: Ask about current AI developments
answer = research_with_gpt(
    "What are the latest developments in AI language models in the past week?"
)
print(answer)

## Example 2: Multi-Step Research

GPT can perform multiple searches to answer complex questions.

In [ ]:
# Example 2: Complex research requiring multiple searches
answer = research_with_gpt(
    "Compare the current pricing of GPT-4o and Claude 3.5 Sonnet, "
    "and summarize the key differences in their capabilities."
)
print(answer)

## Example 3: News Search

Use the news_search tool specifically for recent news articles.

In [ ]:
# Example 3: News-focused research
answer = research_with_gpt(
    "What are the latest news about open-source AI models this week?"
)
print(answer)

## Direct Usage of free-web-search-ultimate

You can also use the package directly without GPT for simple searches.

In [ ]:
from free_web_search.search_web import UltimateSearcher
import json

# Initialize the searcher
_searcher = UltimateSearcher()

def search(query: str, max_results: int = 5) -> list:
    """Search the web for real-time information."""
    answer = _searcher.search(query)
    return [
        {'title': s.title, 'url': s.url, 'snippet': s.snippet}
        for s in answer.sources[:max_results]
    ]

def search_news(query: str, max_results: int = 5) -> list:
    """Search for recent news articles."""
    answer = _searcher.search(query, search_type='news')
    return [
        {'title': s.title, 'url': s.url, 'snippet': s.snippet, 'date': getattr(s, 'date', '')}
        for s in answer.sources[:max_results]
    ]

# Direct web search
results = search("OpenAI GPT-4o latest updates", max_results=3)
for r in results:
    print(f"Title: {r.get('title', 'N/A')}")
    print(f"URL: {r.get('url', r.get('href', 'N/A'))}")
    print(f"Snippet: {r.get('body', r.get('snippet', 'N/A'))[:200]}")
    print()

In [ ]:
# Direct news search
news = search_news("artificial intelligence news", max_results=3)
for n in news:
    print(f"Title: {n.get('title', 'N/A')}")
    print(f"URL: {n.get('url', n.get('href', 'N/A'))}")
    print(f"Date: {n.get('date', 'N/A')}")
    print()

## MCP Server Usage

The `free-web-search-ultimate` package also works as an MCP (Model Context Protocol) server, allowing you to use it with Claude Desktop, Cursor, Windsurf, and other MCP-compatible clients.

### Claude Desktop Configuration

Add this to your `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "free-web-search": {
      "command": "free-web-search-mcp"
    }
  }
}
```

### CLI Usage

```bash
# Search the web
search-web "latest AI news"

# Search for news
search-web --type news "OpenAI announcements"
```

## Summary

In this cookbook, we demonstrated how to:

1. **Install** `free-web-search-ultimate` — a zero-cost, privacy-first web search package
2. **Define tools** for GPT using the OpenAI function calling API
3. **Build an agentic loop** that handles multi-turn tool use
4. **Research complex topics** by combining GPT's reasoning with real-time web search

### Key Benefits

- **Zero additional cost** — no search API subscription needed
- **Privacy-first** — uses DuckDuckGo by default
- **Easy integration** — simple Python API
- **MCP compatible** — works with Claude Desktop and other MCP clients

### Resources

- [free-web-search-ultimate on GitHub](https://github.com/wd041216-bit/free-web-search-ultimate)
- [free-web-search-ultimate on PyPI](https://pypi.org/project/free-web-search-ultimate/)
- [Glama MCP Server](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)
- [OpenAI Function Calling Documentation](https://platform.openai.com/docs/guides/function-calling)